# Text Generation

> Everything to know about causal language models: what next-token prediction actually optimises, how each decoding parameter changes the output, where the time and memory go at inference, what 4-bit quantisation costs, and runnable code that measures all of it on a 12 GB card.

- skip_showdoc: true
- skip_exec: true

## 1. What is Text Generation?

A causal language model estimates $P(x_t \mid x_{<t})$ - the probability of the next token given everything before it - and generation is sampling from that distribution repeatedly, feeding each token back in. Everything an LLM appears to do (answering, translating, coding, reasoning) is this one operation with different prefixes.

**Input.** A token sequence. For an instruction model it is a *formatted* sequence: the chat template turns `[{"role": "user", "content": "..."}]` into the exact special-token layout the model was fine-tuned on. Applying the wrong template, or none, is the most common cause of "the model got much worse after I stopped using the pipeline".

**Output.** One token at a time, autoregressively, until a stop token or a length limit. Which token depends entirely on the **decoding strategy**, and that choice changes the output more than a model upgrade often does.

**Base vs instruct**, a distinction worth being blunt about:

| Model | Trained on | Behaviour | Use for |
|---|---|---|---|
| Base | Raw text, next-token prediction | Continues text; will not follow instructions | Fine-tuning, perplexity, few-shot prompting |
| Instruct / chat | + supervised fine-tuning + preference optimisation | Follows instructions, has a chat template | Everything conversational |
| Reasoning | + RL on verifiable tasks | Emits a long chain of thought before answering | Maths, code, multi-step problems |

**Two phases with completely different cost profiles**, which is the single most useful mental model for LLM inference:

- **Prefill** - process the whole prompt in one parallel forward pass. Compute-bound. Cost scales with prompt length.
- **Decode** - generate one token per forward pass, reusing the KV cache. **Memory-bandwidth-bound**, not compute-bound: the GPU spends its time reading weights, not multiplying. This is why batching is nearly free, why quantisation speeds up generation, and why tokens/second barely improves on a faster-compute card with the same bandwidth.

---

## 2. Real-World Use Cases

| Use case | Domain | Consumes / produces | Dominant constraint |
|---|---|---|---|
| Chat assistants | Consumer, enterprise | Conversation -> reply | Time to first token; safety; cost per conversation |
| Code completion | Developer tools | File prefix/suffix -> completion | Sub-200 ms latency; fill-in-the-middle format; local models for privacy |
| Agents and tool use | Automation | Goal + tool schemas -> tool calls | Structured output validity; error recovery; runaway loops |
| RAG answering | Enterprise search | Question + retrieved chunks -> grounded answer | Faithfulness to context; citation; long context |
| Synthetic data generation | ML teams | Seed prompts -> training data | Diversity; contamination of your own eval sets |
| Content drafting | Marketing, support | Brief -> draft | Brand voice; factual grounding; human review loop |
| Structured extraction | Data engineering | Document + schema -> JSON | Schema validity (use constrained decoding); throughput |
| On-device assistants | Mobile, embedded | Local prompt -> local reply | 1-4B params quantised; memory and battery |

What the benchmark number hides:

- **Serving cost is dominated by decode, and decode is memory-bandwidth-bound.** A 7B model in fp16 must read 14 GB of weights *per token* at batch size 1. That is why real deployments batch aggressively, quantise, and use paged-attention servers rather than a `generate()` loop.
- **Context length is priced twice.** The KV cache grows linearly with sequence length and lives in VRAM alongside the weights, so a long conversation can OOM a model that loaded fine. Section 10 measures it.
- **Benchmarks leak.** Public evaluation sets appear in pretraining corpora. Treat MMLU-style scores as weak evidence and prefer held-out, private evaluations or human preference rankings.
- **The prompt is part of the system.** Template, system message, and decoding parameters change results by more than most model swaps. Version them like code.

---

## 3. How Modern Text Generation Works

1. **N-gram models (1950s-2000s).** Count sequences, back off when unseen. Perplexity as the metric dates from here. No generalisation beyond the counted context.
2. **RNNs and LSTMs (2010-2016).** A learned hidden state instead of counts. Better generalisation, but sequential training and a vanishing memory of the distant past.
3. **The transformer decoder (2017-2019).** Self-attention plus positional encoding, trained in parallel across positions. GPT-2 (2019) showed that scale plus next-token prediction produced surprisingly coherent text.
4. **Scaling (2020-2022).** GPT-3 (175B) demonstrated in-context learning; Chinchilla (2022) corrected the compute-optimal ratio of parameters to tokens, and the field started training smaller models on far more data - which is why a 2026-era 3B model outperforms a 2020-era 13B.
5. **Instruction tuning and preference optimisation (2022-2024).** SFT on demonstrations, then RLHF/DPO on preference pairs. This is the step that turned a text continuer into an assistant, and it changed capability rankings more than another order of magnitude of pretraining would have.
6. **Efficiency as architecture (2023-2026).** Grouped-query attention (smaller KV cache), RoPE scaling for long context, sliding-window attention, and **mixture of experts** - route each token to a few experts so a model with 30B total parameters activates only ~3B per token. FlashAttention made attention IO-aware; speculative decoding uses a small draft model to propose tokens a big model verifies in one pass.
7. **Reasoning models (2024-2026).** Reinforcement learning on verifiable outcomes (maths, code, unit tests) taught models to spend test-time compute on an explicit chain of thought before answering. Qwen3's hybrid `enable_thinking` switch is the practical form: the same weights, thinking on for hard problems and off for latency-sensitive ones.
8. **Small models got good.** The 0.5-4B class - Qwen3, SmolLM3, Gemma 3, Phi-4-mini - now handles summarisation, extraction, classification and routine chat well enough to deploy, which is what makes a 12 GB card a useful LLM machine rather than a toy.

**Where it stands (mid-2026).** Locally on 12 GB VRAM: 1-4B models in fp16, or 7-14B in 4-bit, with a real quality gap to frontier APIs on reasoning-heavy work and near-parity on extraction, classification and summarisation. The active questions are test-time compute (how long to let a model think), agentic reliability, and serving economics rather than raw next-token quality.

---

## 4. Evaluation Metrics

**Perplexity** - the exponentiated mean negative log-likelihood per token:

$$\text{PPL} = \exp\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(x_i \mid x_{<i})\right)$$

Intuitively, the effective number of equally likely choices the model has at each step. Lower is better. It is the only intrinsic metric, and it has three hard caveats:

- **It is tokenizer-dependent.** Different vocabularies split text differently, so perplexities from models with different tokenizers are *not comparable*. Comparing Qwen's PPL to Llama's is comparing nothing.
- **It measures fit to a corpus, not usefulness.** An instruction-tuned model usually scores *worse* perplexity on raw web text than its base model while being far more useful.
- **The stride matters.** Sliding-window evaluation with a large stride gives every token less context and inflates perplexity. Report the window and stride.

**Capability benchmarks:** MMLU (knowledge), GSM8K and MATH (arithmetic reasoning), HumanEval and LiveCodeBench (code), IFEval (instruction following), GPQA (hard science). All are contaminated to unknown degrees. [LMArena](https://lmarena.ai/) human preference and [Artificial Analysis](https://artificialanalysis.ai/) cost/latency comparisons are the practical cross-checks.

**Serving metrics, which decide deployments:**

| Metric | Meaning | Bound by |
|---|---|---|
| TTFT (time to first token) | Prefill latency | Compute, prompt length |
| TPOT / inter-token latency | Time per generated token | **Memory bandwidth** |
| Throughput (tokens/s, total) | Across all concurrent requests | Batching, KV cache capacity |
| VRAM at length L | Weights + KV cache | Model size, context, batch |

The cell below implements sliding-window perplexity and the two throughput measurements, and they are used throughout the notebook.

---

In [ ]:
import time

import torch


@torch.inference_mode()
def perplexity(model, tok, text, window=1024, stride=512):
    "Sliding-window perplexity. Report `window` and `stride` with any number you quote.\n\n    Each pass scores only the last `stride` tokens of a `window`-token context, so every\n    scored token has as much left-context as the window allows.\n    "
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    nlls, count = [], 0
    for begin in range(0, ids.shape[1], stride):
        end = min(begin + window, ids.shape[1])
        target_len = end - begin if begin == 0 else end - (begin + window - stride)
        if target_len <= 0:
            break
        chunk = ids[:, begin:end]
        targets = chunk.clone()
        targets[:, :-target_len] = -100          # only score the fresh tokens
        out = model(chunk, labels=targets)
        n = int((targets != -100).sum())
        nlls.append(out.loss.float() * n)
        count += n
        if end == ids.shape[1]:
            break
    return torch.exp(torch.stack(nlls).sum() / count).item()


@torch.inference_mode()
def timed_generate(model, tok, prompt, max_new_tokens=128, **gen_kwargs):
    "Generate and report TTFT (prefill), decode rate, and the text.\n\n    TTFT is measured with a 1-token generation, which is the prefill cost plus one decode\n    step - close enough to prefill for comparison, and it needs no callback plumbing.\n    "
    enc = tok(prompt, return_tensors="pt").to(model.device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    model.generate(**enc, max_new_tokens=1, do_sample=False, pad_token_id=tok.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    ttft = time.perf_counter() - t0

    t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=max_new_tokens,
                         pad_token_id=tok.eos_token_id, **gen_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    new_tokens = out.shape[1] - enc["input_ids"].shape[1]
    return {
        "text": tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True),
        "prompt_tokens": enc["input_ids"].shape[1],
        "new_tokens": new_tokens,
        "ttft_s": round(ttft, 3),
        "tokens_per_sec": round(new_tokens / elapsed, 1),
        "total_s": round(elapsed, 2),
    }


def kv_cache_gb(layers, kv_heads, head_dim, seq_len, batch=1, bytes_per=2):
    "VRAM held by the KV cache: 2 (K and V) x layers x kv_heads x head_dim x length x batch."
    return 2 * layers * kv_heads * head_dim * seq_len * batch * bytes_per / 1e9


# The KV cache arithmetic, before loading anything. Qwen3-1.7B: 28 layers, 8 KV heads
# (grouped-query attention), head_dim 128.
for length in [1024, 8192, 32768]:
    print(f"  KV cache at {length:6d} tokens, batch 1: "
          f"{kv_cache_gb(28, 8, 128, length):5.2f} GB   "
          f"batch 8: {kv_cache_gb(28, 8, 128, length, batch=8):5.2f} GB")
print("\n(grouped-query attention is why these numbers are survivable: with 16 full KV heads\n"
      " instead of 8, every figure above doubles)")

## 5. Datasets

| Dataset | Contents | Size | Scope | License | Typical use |
|---|---|---|---|---|---|
| [FineWeb / FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb) | Filtered CommonCrawl web text | 15T / 1.3T tokens | en (+ FineWeb-2 multilingual) | ODC-By | Modern open pretraining corpus |
| [The Pile](https://huggingface.co/datasets/EleutherAI/pile) | 22 mixed sources (code, papers, books) | 825 GB | en | mixed | The 2020-2022 standard; historically important |
| [Dolma](https://huggingface.co/datasets/allenai/dolma) | Open, documented pretraining mix | 3T tokens | en | ODC-By | Fully transparent pretraining |
| [WikiText-2 / 103](https://huggingface.co/datasets/Salesforce/wikitext) | Clean Wikipedia articles | 2M / 100M tokens | en | CC BY-SA 3.0 | **Perplexity**; used below |
| [Tulu 3 SFT mixture](https://huggingface.co/datasets/allenai/tulu-3-sft-mixture) | Instruction demonstrations | 940k | en | ODC-By | Instruction tuning |
| [UltraFeedback](https://huggingface.co/datasets/openbmb/UltraFeedback) | Preference pairs | 64k | en | MIT | DPO / preference optimisation |
| [MMLU](https://huggingface.co/datasets/cais/mmlu) | 57 subjects, multiple choice | 16k | en | MIT | Knowledge benchmark (contaminated) |
| [GSM8K](https://huggingface.co/datasets/openai/gsm8k) | Grade-school maths word problems | 8.5k | en | MIT | Arithmetic reasoning |
| [HumanEval](https://huggingface.co/datasets/openai/openai_humaneval) | Python problems + unit tests | 164 | en | MIT | Code generation (small, saturated) |
| [IFEval](https://huggingface.co/datasets/google/IFEval) | Verifiable instruction constraints | 500 | en | Apache 2.0 | Instruction following, hard to game |

This notebook measures perplexity on **WikiText-2-raw test** (a 0.7 MB parquet, read directly rather than through the dataset's 640 MB repo). It is clean encyclopedic English - a fine instrument for comparing two models with the *same* tokenizer, and meaningless across tokenizers.

---

## 6. The Model Landscape (mid-2026)

Rankings worth reading: [LMArena](https://lmarena.ai/) (human preference), [Artificial Analysis](https://artificialanalysis.ai/) (quality vs price vs speed), [Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) (automated benchmarks).

What fits on this box (12 GB VRAM, 20 GB RAM):

| Model | Params | License | Context | fp16 VRAM | Notes |
|---|---|---|---|---|---|
| [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) | 0.6B | Apache 2.0 | 40k | ~1.4 GB | classification, extraction, drafts |
| [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) | 1.7B | Apache 2.0 | 40k | ~3.5 GB | the notebook's workhorse; hybrid thinking |
| [SmolLM3-3B](https://huggingface.co/HuggingFaceTB/SmolLM3-3B) | 3B | Apache 2.0 | 64k+ | ~6.2 GB | fully open training recipe; used below |
| [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) | 4B | Apache 2.0 | 32k+ | ~8 GB | strongest that fits comfortably in fp16 |
| [Gemma-3-4B-it](https://huggingface.co/google/gemma-3-4b-it) | 4B | Gemma terms | 128k | ~8 GB | long context, multimodal variant (gated) |
| [Phi-4-mini-instruct](https://huggingface.co/microsoft/Phi-4-mini-instruct) | 3.8B | MIT | 128k | ~7.6 GB | strong reasoning per parameter |
| [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) | 8B | Apache 2.0 | 128k | ~16 GB | **4-bit only** here (~5.5 GB) |
| [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) | 8B | Llama 3.1 | 128k | ~16 GB | gated; 4-bit only here |
| [Qwen3-30B-A3B](https://huggingface.co/Qwen/Qwen3-30B-A3B) | 30B (3B active) | Apache 2.0 | 128k | ~60 GB | MoE: fast per token, does not fit |

**Sizing rules for a 12 GB card.** Weights in bytes = params x 2 (fp16) or x ~0.55 (4-bit NF4, including overhead). Then add the KV cache from section 4, then leave ~1 GB of headroom for activations. A 7-8B model in 4-bit plus an 8k context is comfortable; the same model in fp16 is not.

**The download caveat.** 4-bit quantisation happens *after* the full fp16/bf16 weights are downloaded, so `load_in_4bit` saves VRAM, not bandwidth or disk. An 8B model is a ~16 GB download whatever you load it as - past this repo's ~8 GB runnable budget, which is why the sections below stop at 3B.

---

## 7. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - the models, `generate`, and `TextStreamer`
- `accelerate` - `device_map` placement
- `bitsandbytes` - 4-bit quantisation in section 11
- `datasets` - the WikiText-2 slice for perplexity
- `pandas` + `pyecharts` - benchmark table and charts

`pipeline("text-generation")` still exists in transformers v5, but everything below calls `AutoModelForCausalLM` and `generate` directly - the point of the notebook is the parameters the pipeline hides.

**Memory discipline matters more here than in any other notebook in this folder.** The models are the largest thing in the folder, and a 3B model plus a stale 1.7B model plus a KV cache will exceed 12 GB. Every section frees its model before the next one loads.

---

In [ ]:
# Everything runs through Hugging Face transformers - no vLLM, no vendor runtimes.
# %pip install -q torch transformers accelerate bitsandbytes datasets pandas pyecharts

In [ ]:
import ctypes
import ctypes.util
import gc
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
if device != "cpu":
    print(torch.cuda.get_device_name(0))
    print(f"total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("device:", device, "| dtype:", dtype)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:22s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "Collect garbage and hand freed VRAM back to the CUDA allocator.\n\n    Call right after `del`-ing a model you are done with: `del model; free_memory()`.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    # glibc keeps freed CPU allocations in its arenas instead of returning them to the
    # OS, so RSS compounds across sections. malloc_trim(0) hands the arenas back.
    try:
        ctypes.CDLL(ctypes.util.find_library("c") or "libc.so.6").malloc_trim(0)
    except Exception:
        pass


# All downloads go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)
HF_CACHE = str(DATA_DIR / "hf_cache")

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# WikiText-2-raw test, read directly from its parquet (the repo holds wikitext-103 too).
wiki = load_dataset(
    "parquet",
    data_files={"test": "hf://datasets/Salesforce/wikitext/wikitext-2-raw-v1/test-*.parquet"},
    split="test",
    cache_dir=HF_CACHE,
)
WIKI_TEXT = "\n\n".join(t for t in wiki["text"] if t.strip())[:200_000]
print(f"perplexity corpus: {len(WIKI_TEXT):,} characters")

MODEL_ID = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=dtype, device_map=device, cache_dir=HF_CACHE
).eval()
vram("qwen3-1.7b loaded")

cfg = model.config
print(f"\n{MODEL_ID}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params, "
      f"{cfg.num_hidden_layers} layers, {cfg.num_attention_heads} attention heads, "
      f"{getattr(cfg, 'num_key_value_heads', cfg.num_attention_heads)} KV heads, "
      f"vocab {cfg.vocab_size:,}")

## 8. Decoding: the parameters that change everything

The model gives a probability distribution; the decoding strategy chooses from it. These parameters do more to the output than most model upgrades.

| Strategy | Parameter | What it does | Use for |
|---|---|---|---|
| Greedy | `do_sample=False` | Always the argmax | Extraction, classification, anything deterministic |
| Beam search | `num_beams=4` | Keeps k partial sequences, picks the best total probability | Translation, summarisation - **not** open chat |
| Temperature | `temperature` | Sharpens (<1) or flattens (>1) the distribution before sampling | The main creativity dial |
| Top-k | `top_k=50` | Sample only from the k likeliest tokens | Cheap truncation of the tail |
| Top-p (nucleus) | `top_p=0.9` | Sample from the smallest set whose mass exceeds p | The standard: adapts to how peaked the distribution is |
| Min-p | `min_p=0.05` | Keep tokens above a fraction of the top token's probability | Robust at high temperature |
| Repetition penalty | `repetition_penalty=1.1` | Down-weights tokens already produced | Fixing loops; too high damages fluency |
| No-repeat n-gram | `no_repeat_ngram_size=3` | Forbids repeating any trigram | Blunt but effective for summarisation |

**Greedy decoding on open-ended text degenerates into repetition.** Not a bug in the implementation - a consequence of always taking the most likely token, which makes a self-reinforcing loop the highest-probability continuation. That is why sampling exists, and the cell below shows it happening.

The right default depends entirely on the job: `do_sample=False` when you want the same answer every time (extraction, classification, SQL), and `temperature=0.7, top_p=0.9` when you want text a human enjoys reading.

---

In [ ]:
PROMPT = "The three most important things to know about the ocean are"

configs = [
    ("greedy", dict(do_sample=False)),
    ("beam search (4)", dict(do_sample=False, num_beams=4)),
    ("temp 0.3", dict(do_sample=True, temperature=0.3, top_p=0.95)),
    ("temp 0.7 + top_p 0.9", dict(do_sample=True, temperature=0.7, top_p=0.9)),
    ("temp 1.5", dict(do_sample=True, temperature=1.5, top_p=0.95)),
    ("greedy + rep penalty", dict(do_sample=False, repetition_penalty=1.15)),
]

torch.manual_seed(0)
for name, kwargs in configs:
    r = timed_generate(model, tok, PROMPT, max_new_tokens=60, **kwargs)
    print(f"--- {name} ({r['tokens_per_sec']} tok/s) ---\n{r['text'].strip()[:300]}\n")

# What the distribution actually looks like at one position - the thing all of the above
# are manipulating.
enc = tok(PROMPT, return_tensors="pt").to(model.device)
with torch.inference_mode():
    logits = model(**enc).logits[0, -1].float()
probs = torch.softmax(logits, dim=-1)
top = probs.topk(10)
print("next-token distribution:")
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"   {p:6.3f}  {tok.decode([i])!r}")
cumulative = torch.cumsum(probs.sort(descending=True).values, 0)
print(f"\ntokens needed to cover 90% of the mass (top_p=0.9): "
      f"{int((cumulative < 0.9).sum()) + 1} of {len(probs):,}")

## 9. Chat templates, system prompts, thinking, and streaming

An instruction model is fine-tuned on a **specific token layout** - special tokens marking roles, turns and generation starts. `tokenizer.apply_chat_template` reproduces it exactly. Feed the model a bare string instead and quality drops for reasons that never show up as an error.

Three things this section makes visible:

- **What the template actually emits.** Printed raw below, once, because seeing `<|im_start|>assistant` explains more than any description of chat formatting.
- **Thinking mode.** Qwen3 is a hybrid reasoning model: `enable_thinking=True` makes it emit a `<think>...</think>` block before answering. It measurably improves multi-step problems and costs hundreds of extra tokens - a latency/quality dial rather than a free win. Off is the right default for extraction and classification, on for maths and planning.
- **Streaming.** `TextStreamer` prints tokens as they arrive. It changes no output, only the perceived latency, and perceived latency is most of what users mean by "fast".

---

In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "system", "content": "You are a terse assistant. Answer in one sentence."},
    {"role": "user", "content": "Why is the KV cache the thing that limits context length?"},
]

formatted = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
print("--- what the model actually sees ---")
print(repr(formatted))

print("\n--- streamed answer (thinking off) ---")
enc = tok(formatted, return_tensors="pt").to(model.device)
streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)
with torch.inference_mode():
    model.generate(**enc, max_new_tokens=120, do_sample=False,
                   pad_token_id=tok.eos_token_id, streamer=streamer)

# Thinking on vs off, on a problem where it matters, with the token cost attached.
problem = ("A shop sells pens at 3 for 5 dollars and notebooks at 2 for 7 dollars. "
           "If I spend exactly 31 dollars and buy 9 pens, how many notebooks did I buy?")
for thinking in [False, True]:
    chat = tok.apply_chat_template([{"role": "user", "content": problem}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=thinking)
    r = timed_generate(model, tok, chat, max_new_tokens=400 if thinking else 120, do_sample=False)
    print(f"\n--- thinking={thinking}: {r['new_tokens']} tokens, {r['total_s']}s, "
          f"{r['tokens_per_sec']} tok/s ---")
    print(r["text"].strip()[:700])

## 10. Where the time and the memory go

Two measurements that explain most of LLM serving.

**Prefill vs decode.** Prefill processes the whole prompt in one parallel pass; decode does one token per pass. Below, TTFT grows with prompt length while the decode rate stays roughly flat - because decode re-reads the same weights every step regardless of how long the context is. Decode is bandwidth-bound: at batch size 1 the GPU reads ~3.5 GB of weights per token and does very little arithmetic with them.

**The KV cache.** Every generated token appends a key and value vector per layer per KV head, and it all lives in VRAM. It is why a model that loads in 4 GB can OOM at 32k context, and why grouped-query attention (fewer KV heads than attention heads) was adopted so universally - it cuts this by the head ratio directly.

**Batching is nearly free** in the same regime: several sequences read the same weights in one pass, so total throughput rises far faster than latency does. This is the entire economic argument for a batching server (vLLM, TGI, SGLang) over a `generate()` loop.

---

In [ ]:
import pandas as pd

# Prefill vs decode, across prompt lengths.
rows = []
for n_words in [16, 128, 512, 2048]:
    prompt = ("The history of computing is long and varied. " * 400).split()[:n_words]
    prompt = " ".join(prompt)
    r = timed_generate(model, tok, prompt, max_new_tokens=64, do_sample=False)
    rows.append({"prompt_tokens": r["prompt_tokens"], "ttft_s": r["ttft_s"],
                 "decode_tok_per_s": r["tokens_per_sec"]})
    print(rows[-1])

# Batched decode: the same weight reads amortised over more sequences.
batch_rows = []
prompt = "Explain in detail why memory bandwidth limits token generation."
enc = tok([prompt] * 8, return_tensors="pt").to(model.device)
for bs in [1, 2, 4, 8]:
    sub = {k: v[:bs] for k, v in enc.items()}
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**sub, max_new_tokens=64, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    total_new = (out.shape[1] - sub["input_ids"].shape[1]) * bs
    batch_rows.append({"batch": bs, "seconds": round(elapsed, 2),
                       "total_tok_per_s": round(total_new / elapsed, 1),
                       "per_seq_tok_per_s": round(total_new / elapsed / bs, 1)})
    print(batch_rows[-1])

vram("after throughput tests")
pd.DataFrame(batch_rows)

In [ ]:
from pyecharts import options as opts
from pyecharts.charts import Line

line = (
    Line()
    .add_xaxis([str(r["batch"]) for r in batch_rows])
    .add_yaxis("total tokens/s", [r["total_tok_per_s"] for r in batch_rows], is_smooth=True)
    .add_yaxis("tokens/s per sequence", [r["per_seq_tok_per_s"] for r in batch_rows], is_smooth=True)
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="Batching is nearly free in the decode phase",
            subtitle="Qwen3-1.7B fp16 on an RTX 3060 - weights are read once per step, not once per sequence",
        ),
        xaxis_opts=opts.AxisOpts(name="batch size"),
        yaxis_opts=opts.AxisOpts(name="tokens / second"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
    )
)
line.render_notebook()

## 11. Quantisation: what 4-bit actually costs

Quantisation stores weights in fewer bits. NF4 (the 4-bit format in `bitsandbytes`) cuts VRAM roughly 3.5x versus fp16 and usually *speeds up* generation, because decode is bandwidth-bound and there is less to read - the dequantisation arithmetic is cheaper than the memory it saves.

What it costs is quality, and this section measures it rather than asserting it: perplexity on the same WikiText slice, fp16 against 4-bit, same model, same tokenizer, so the comparison is legitimate.

**Three things to keep straight:**

- **`load_in_4bit` does not shrink the download.** The full bf16 repo is fetched and quantised on load. To download less you need a repo that *stores* quantised weights (GPTQ/AWQ).
- **`bnb_4bit_compute_dtype` is separate from the storage dtype.** Weights are stored in 4 bits and dequantised to fp16 for the matmul.
- **Double quantisation** (`bnb_4bit_use_double_quant=True`) quantises the quantisation constants too - a further ~0.4 bits per parameter, free.

The rule of thumb this section exists to make concrete: **a larger model at 4 bits usually beats a smaller model at fp16 for the same VRAM.** An 8B in 4-bit (~5.5 GB) against a 3B in fp16 (~6.2 GB) is the real choice on a 12 GB card - it is not run here only because the 8B download exceeds the repo's budget.

---

In [ ]:
from transformers import BitsAndBytesConfig

PPL_TEXT = WIKI_TEXT[:60_000]   # keep the perplexity run to a couple of minutes

t0 = time.perf_counter()
ppl_fp16 = perplexity(model, tok, PPL_TEXT, window=1024, stride=512)
fp16_ppl_seconds = time.perf_counter() - t0
fp16_gen = timed_generate(model, tok, PROMPT, max_new_tokens=64, do_sample=False)
fp16_vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else float("nan")
print(f"fp16 : ppl {ppl_fp16:6.2f}  {fp16_gen['tokens_per_sec']:6.1f} tok/s  "
      f"{fp16_vram:5.2f} GB  ({fp16_ppl_seconds:.0f}s to score)")

del model
free_memory()
vram("freed fp16")

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # normal-float 4, better than plain int4 for weights
    bnb_4bit_compute_dtype=torch.float16,   # dequantised to fp16 for the matmul
    bnb_4bit_use_double_quant=True,         # quantise the quantisation constants too
)
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=quant_cfg, device_map=device, cache_dir=HF_CACHE
).eval()
vram("4-bit loaded")

ppl_4bit = perplexity(model_4bit, tok, PPL_TEXT, window=1024, stride=512)
q4_gen = timed_generate(model_4bit, tok, PROMPT, max_new_tokens=64, do_sample=False)
q4_vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else float("nan")
print(f"4-bit: ppl {ppl_4bit:6.2f}  {q4_gen['tokens_per_sec']:6.1f} tok/s  {q4_vram:5.2f} GB")
print(f"\nVRAM {fp16_vram / max(q4_vram, 1e-9):.1f}x smaller, perplexity "
      f"{100 * (ppl_4bit - ppl_fp16) / ppl_fp16:+.1f}%")

quant_results = [
    {"model": "qwen3-1.7b fp16", "perplexity": round(ppl_fp16, 2),
     "tok_per_s": fp16_gen["tokens_per_sec"], "vram_gb": round(fp16_vram, 2)},
    {"model": "qwen3-1.7b 4-bit", "perplexity": round(ppl_4bit, 2),
     "tok_per_s": q4_gen["tokens_per_sec"], "vram_gb": round(q4_vram, 2)},
]

del model_4bit
free_memory()
vram("after quantisation section")

## 12. Head-to-head Benchmark

Three configurations on the same corpus and the same prompt: a 0.6B, a 1.7B, and a 3B, plus the 4-bit result from section 11. One model live at a time.

**The perplexity column is only partly comparable.** Qwen3-0.6B and Qwen3-1.7B share a tokenizer, so their numbers can be compared directly. SmolLM3-3B has a different vocabulary, so its perplexity is on a different scale and belongs in the table only as a within-model reference - which is exactly the caveat from section 4, made concrete instead of stated.

What is comparable across all of them: **VRAM, tokens per second, and TTFT**. Those are the columns that decide what you can actually run on a 12 GB card, and they are the reason this notebook measures rather than cites.

---

In [ ]:
BENCH = [
    ("qwen3-0.6b", "Qwen/Qwen3-0.6B"),
    ("qwen3-1.7b", "Qwen/Qwen3-1.7B"),
    ("smollm3-3b", "HuggingFaceTB/SmolLM3-3B"),
]

results = []
for name, model_id in BENCH:
    bench_tok = AutoTokenizer.from_pretrained(model_id, cache_dir=HF_CACHE)
    bench_model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=dtype, device_map=device, cache_dir=HF_CACHE
    ).eval()
    weights_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else float("nan")

    gen = timed_generate(bench_model, bench_tok, PROMPT, max_new_tokens=64, do_sample=False)
    ppl = perplexity(bench_model, bench_tok, WIKI_TEXT[:40_000], window=1024, stride=512)
    results.append({
        "model": name,
        "params_b": round(sum(p.numel() for p in bench_model.parameters()) / 1e9, 2),
        "vram_gb": round(weights_gb, 2),
        "perplexity": round(ppl, 2),
        "ttft_s": gen["ttft_s"],
        "tok_per_s": gen["tokens_per_sec"],
        "tokenizer_vocab": bench_model.config.vocab_size,
    })
    print(results[-1])
    del bench_model, bench_tok   # free before loading the next so VRAM stays flat
    free_memory()

vram("after benchmark")
df = pd.DataFrame(results)
print("\nperplexity is comparable only within a tokenizer family - check the vocab column")
df

In [ ]:
from pyecharts.charts import Bar

bar = (
    Bar()
    .add_xaxis([r["model"] for r in results])
    .add_yaxis("tokens / second", [r["tok_per_s"] for r in results])
    .add_yaxis("VRAM (GB) x10", [round(r["vram_gb"] * 10, 1) for r in results])
    .add_yaxis("perplexity", [r["perplexity"] for r in results])
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="Local LLMs on an RTX 3060 (12 GB)",
            subtitle="fp16, batch 1, 64 new tokens - perplexity comparable only within a tokenizer",
        ),
        yaxis_opts=opts.AxisOpts(name="value"),
        xaxis_opts=opts.AxisOpts(name="model"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
)
bar.render_notebook()

In [ ]:
from pyecharts.charts import Scatter

# The chart that decides a local deployment: speed against memory.
points = results + [{"model": q["model"], "tok_per_s": q["tok_per_s"], "vram_gb": q["vram_gb"]}
                    for q in quant_results if "4-bit" in q["model"]]
scatter = Scatter()
scatter.add_xaxis([p["vram_gb"] for p in points])
for p in points:
    scatter.add_yaxis(
        p["model"], [[p["vram_gb"], p["tok_per_s"]]],
        symbol_size=18, label_opts=opts.LabelOpts(is_show=False),
    )
scatter.set_global_opts(
    title_opts=opts.TitleOpts(title="Generation speed vs VRAM"),
    xaxis_opts=opts.AxisOpts(name="VRAM for weights (GB)", type_="value"),
    yaxis_opts=opts.AxisOpts(name="tokens / second", type_="value"),
    tooltip_opts=opts.TooltipOpts(trigger="item"),
)
scatter.render_notebook()

## 13. Interactive: a streaming chat loop with history

A minimal multi-turn chat: a message list, the chat template, streaming output, and the KV cost of the conversation printed as it grows. This is the cell people run on its own, so it opens with a `require(...)` guard naming what it needs from Setup rather than dying on a bare `NameError`.

Two things it makes tangible. **History is quadratic in cost**, not linear: every turn re-prefills the whole conversation, so turn 10 costs far more than turn 1 - which is why production chat servers cache the prefix rather than re-encoding it. And **the system prompt is the cheapest behaviour control you have**: change one line below and the model's register, length and format change more than a parameter swap would.

Edit `TURNS`, or wire it to `input()` for a real loop.

---

In [ ]:
def require(*names):
    "Fail early and clearly if the notebook's setup / helper cells have not been run."
    missing = [n for n in names if n not in globals()]
    if missing:
        raise NameError(
            f"this demo needs {', '.join(missing)} from earlier in the notebook. "
            "Run the setup and helper cells first (Run > Run All Above Selected Cell)."
        )


require("device", "dtype", "HF_CACHE", "free_memory", "vram")

import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

SYSTEM = "You are a concise engineering assistant. Prefer bullet points. Never apologise."
TURNS = [
    "What is a KV cache, in two sentences?",
    "How does grouped-query attention change its size?",
    "Given 12 GB of VRAM and a 3B model in fp16, roughly what context length fits?",
]
GEN = dict(max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9)

# Re-runnable: this cell frees the model at the end, so guard the load or a second
# shift-enter raises NameError.
if "chat_model" not in globals():
    chat_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B", cache_dir=HF_CACHE)
    chat_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-1.7B", dtype=dtype, device_map=device, cache_dir=HF_CACHE
    ).eval()
    vram("live model")

history = [{"role": "system", "content": SYSTEM}]
for turn, user_message in enumerate(TURNS, 1):
    history.append({"role": "user", "content": user_message})
    prompt = chat_tok.apply_chat_template(history, tokenize=False, add_generation_prompt=True,
                                          enable_thinking=False)
    enc = chat_tok(prompt, return_tensors="pt").to(chat_model.device)

    print(f"\n=== turn {turn} | context {enc['input_ids'].shape[1]} tokens ===")
    print(f"user: {user_message}\nassistant: ", end="")
    streamer = TextStreamer(chat_tok, skip_prompt=True, skip_special_tokens=True)
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = chat_model.generate(**enc, pad_token_id=chat_tok.eos_token_id,
                                  streamer=streamer, **GEN)
    reply = chat_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": reply})
    print(f"[{out.shape[1] - enc['input_ids'].shape[1]} tokens in "
          f"{time.perf_counter() - t0:.1f}s; conversation now {out.shape[1]} tokens]")

del chat_model, chat_tok
free_memory()
vram("final")

## 14. Going Further

- **Do not serve with `generate()`.** vLLM, TGI or SGLang add continuous batching, paged attention and prefix caching, and are typically 5-20x the throughput of a naive loop under concurrency. `generate()` is for experiments; a server is for serving.
- **Constrain structured output.** For JSON, use a grammar/schema-constrained decoder (Outlines, XGrammar, or `transformers`' logits processors) rather than prompting and parsing. It makes invalid output impossible instead of unlikely.
- **Speculative decoding** runs a small draft model to propose k tokens that the large model verifies in one pass - typically 1.5-3x faster with *identical* output distribution. `generate(assistant_model=...)` does it in `transformers`.
- **Fine-tune with LoRA, not full weights.** `peft` + 4-bit base (QLoRA) fine-tunes a 3-8B model on a single 12 GB card. A few thousand examples is enough to fix format compliance and house style, which is what most "the model is not good enough" complaints actually are.
- **Prefer RAG to fine-tuning for knowledge.** Fine-tuning teaches form; retrieval supplies facts. Mixing them up produces a model that confidently states outdated things in the right format - see `07_Feature_Extraction` and `11_Text_Ranking`.
- **Evaluate on held-out, private data.** Public benchmarks are contaminated. Build 50-200 examples from your own traffic with rubric-based grading (LLM-as-judge with a fixed rubric, plus human spot checks) and treat that as ground truth.
- **Budget the KV cache before the weights.** Weights are a constant; KV grows with context and batch, and it is what OOMs a production server at 3am. The `kv_cache_gb` helper in section 4 is worth keeping.
- **Related notebooks.** `09_Fill_Mask` (the bidirectional counterpart to causal LM), `06_Summarization` and `05_Translation` (generation with a source to be faithful to), `02_Table_Question_Answering` (constrained generation into SQL), `07_Feature_Extraction` and `11_Text_Ranking` (the retrieval half of RAG), `Multimodal/01_Image_Text_to_Text` (the same decoder with a vision encoder attached).

---